In [ ]:
#############  PHASE 1  ###############
#####STEP 4: Flood trends analysis- EM-DAT data

##Purpose
#This notebook analyses historical flood disaster records obtained from the EM-DAT database 
#to quantify changes in flood frequency, affected populations and economic damages.
#The objective is to use the EM-DAT disaster database to answer Q2 (has flooding increased?) 
#and Q3 (how far does damage reach?) by analysing flood event frequency, deaths, people affected, and economic damage over time for both countries.

##Research Context
#Understanding historical flood impacts provides the foundation for linking climatic and urbanisation trends with observed disaster outcomes.

##Input Data
#EM-DAT flood records


##Methods
#Flood event summaries
#Damage trend analysis
#Population affected analysis
#Time series visualisation

##Outputs
#Flood frequency plots
#Economic damage trends
#Population impact summaries



#Step 4a- Load and explore the EM-DAT data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats   #for linear regression on trends
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")

BASE_DIR = r"C:\Users\drsih\Desktop\profile\flooding_project"

EMDAT_DIR  = os.path.join(BASE_DIR, "data", "raw", "emdat")
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "outputs", "figures")

#Load EM-DAT file
emdat_raw = pd.read_excel(
    os.path.join(EMDAT_DIR, "emdat_flood_events_SA_Nigeria_1900_2024.xlsx")
)


#First looking at the data
print("Shape:", emdat_raw.shape)
print("\nColumn names:")
print(list(emdat_raw.columns))
print("\nFirst 3 rows:")
print(emdat_raw.head(3))

In [ ]:
print(emdat_raw.columns.tolist())

In [ ]:
#STEP 4b-Clean and standardise the EM-DAT data
#Rename columns to simpler names
#EM-DAT uses verbose column names. We rename for convenience.
#In R: rename() from dplyr
# n Python: df.rename(columns={...})

#Standard EM-DAT column names (adjust if yours differ slightly):
column_map = {
    "Country": "Country",
    "Disaster Type": "Disaster_type",
    "Disaster Subtype": "Subtype",
    "Start Year": "Year",
    "Total Deaths": "Deaths",
    "No. Injured": "Injured",
    "No. Affected": "Affected",
    "No. Homeless": "Homeless",
    "Total Affected": "Total_affected",
    "Reconstruction Costs ('000 US$)": "Reconstruction_cost_000usd",
    "Insured Damage ('000 US$)": "Insured_damage_000usd",
    "Total Damage ('000 US$)": "Total_damage_000usd",
    "CPI": "CPI"
}
#Only rename columns that actually exist in your file
emdat = emdat_raw.rename(columns=column_map).copy()

#Keep only the variables you used
columns_to_keep = [
    "DisNo.",
    "Country",
    "Location",
    "Year",
    "Disaster_type",
    "Subtype",
    "Deaths",
    "Total_affected",
    "Total_damage_000usd"
]

emdat = emdat[columns_to_keep]

In [ ]:
print(emdat.columns)

In [ ]:
#Filter to floodsonly
#EM-DAT contains all disaster types — we only want floods
#In R: filter(Disaster_type == "Flood")
emdat = emdat[emdat["Disaster_type"] == "Flood"].copy()

# Countries of interest only
emdat = emdat[
    emdat["Country"].isin(["Nigeria", "South Africa"])
].copy()

#Add City labels
country_to_city = {
    "Nigeria": "Lagos",
    "South Africa": "Durban"
}

emdat["City"] = emdat["Country"].map(country_to_city)

#Convert damage to millions USD
#EM-DAT stores damage in thousands USD ('000 US$)
#Divide by 1000 to get millions
# Create damage variable if missing
if "Damage_million_usd" not in emdat.columns:
    emdat["Damage_million_usd"] = (
        pd.to_numeric(emdat["Total_damage_000usd"], errors="coerce")
        .fillna(0)
        / 1000
    )


print(emdat.columns)

In [ ]:
# Post-1950
emdat = emdat[emdat["Year"] >= 1950].copy()

#Keep only post-1950 for consistency with climate data 
for col in [
    "Deaths",
    "Injured",
    "Affected",
    "Homeless",
    "Total_affected",
    "Damage_million_usd"
]:
    if col in emdat.columns:
        emdat[col] = emdat[col].fillna(0)

print(f"Flood events: {len(emdat)}")
print(emdat["Country"].value_counts())

In [ ]:
print(emdat["Country"].value_counts())
print(emdat["Year"].min(), emdat["Year"].max())
print(emdat.columns)

In [ ]:
#Create the Decade variable
emdat["Decade"] = (emdat["Year"] // 10) * 10
from pathlib import Path

PROJECT_ROOT = Path("..")
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

# Create the folder if it doesn't exist
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset
emdat.to_csv(PROCESSED_DATA / "emdat_processed.csv", index=False)

print("✓ Cleaned EM-DAT dataset saved successfully.")

In [ ]:
start_year = emdat["Year"].min()
end_year = emdat["Year"].max()

all_years = pd.DataFrame({
    "Year": list(range(start_year, end_year + 1)) * 2,
    "City": ["Lagos"] * (end_year - start_year + 1) +
            ["Durban"] * (end_year - start_year + 1)
})

In [ ]:
#STEP 4c- Event frequency analysis: has flooding increased?

#Count flood events per year per country
#groupby + size() counts rows per group
#In R: group_by(Country, Year) |> summarise(n = n())
freq = (
    emdat
    .groupby(["Year", "City"])
    .size() #count events per year-city group
    .reset_index(name="Flood_events")   #name the count column
)

#Create complete year sequence
start_year = emdat["Year"].min()
end_year = emdat["Year"].max()


#Fill years with zero floods
#Some years have no events, they won't appear in the groupby
#We need to fill them with 0 so the time series is complete
#In R: complete(Year, City, fill = list(Flood_events = 0))

all_years = pd.DataFrame({
    "Year": list(range(start_year, end_year + 1)) * 2,
    "City": ["Lagos"] * (end_year - start_year + 1) +
            ["Durban"] * (end_year - start_year + 1)
})

freq = all_years.merge(
    freq,
    on=["Year", "City"],
    how="left"
)

freq["Flood_events"] = freq["Flood_events"].fillna(0)

#Add decade column
freq["Decade"] = (freq["Year"] // 10) * 10

# Decade totals
freq_decade = (
    freq
    .groupby(["Decade", "City"])["Flood_events"]
    .sum()
    .reset_index(name="Total_events")
)

print("\n=== FLOOD EVENTS BY DECADE ===")

freq_pivot = freq_decade.pivot(
    index="Decade",
    columns="City",
    values="Total_events"
)

print(freq_pivot)


#Plot- annual flood events 

colors = {
    "Lagos": "#378ADD",
    "Durban": "#1D9E75"
}

fig, axes = plt.subplots(
    2,
    1,
    figsize=(12,8),
    sharex=True
)
#sharex=True means both plots share the same x axis (Year)

for i, city in enumerate(["Lagos","Durban"]):

    d = freq[freq["City"] == city]
    
    #Bar chart of annual events
    axes[i].bar(
        d["Year"],
        d["Flood_events"],
        color=colors[city],
        alpha=0.7
    )
    #10-year rolling average to show trend
    #rolling(10).mean() = rollmean(x, 10) in R
    rolling = (
        d
        .set_index("Year")["Flood_events"]
        .rolling(10, min_periods=5)
        .mean()
    )

    axes[i].plot(
        rolling.index,
        rolling.values,
        color="black",
        linewidth=2,
        label="10-year rolling mean"
    )

    axes[i].set_title(f"{city} Flood Frequency")
    axes[i].set_ylabel("Events")
    axes[i].legend()

plt.xlabel("Year")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "flood_frequency_annual.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#STEP 4d-Flood impact analysis (Deaths, Affected People and Economic Damage)

emdat["Decade"] = (emdat["Year"] // 10) * 10

#Aggregate key impact metrics by decade and city
impact_cols = [
    "Deaths",
    "Total_affected",
    "Damage_million_usd"
]
#Check which columns exist
impact_cols = [
    c for c in impact_cols
    if c in emdat.columns
]

impact_decade = (
    emdat
    .groupby(["Decade","City"])[impact_cols]
    .sum()   #total per decade (sum of all events)
    .reset_index()
)

print("\n=== FLOOD IMPACT BY DECADE ===")
print(impact_decade)



In [ ]:
#Plot-3-panel impact summary (deaths, people affected and damaged)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18,5)
)

metrics = [
    "Deaths",
    "Total_affected",
    "Damage_million_usd"
]

titles = [
    "Deaths",
    "People Affected",
    "Economic Damage (Million USD)"
]

for ax, metric, title in zip(
    axes,
    metrics,
    titles
):

    if metric not in impact_decade.columns:
        continue

    for city in ["Lagos","Durban"]:

        d = impact_decade[
            impact_decade["City"] == city
        ]

        values = d[metric]

        if metric == "Total_affected":
            values = values / 1e6

        ax.bar(
            d["Decade"] + (3 if city=="Lagos" else -3),
            #offset bars so they don't overlap — Lagos shifted right, Durban left
            values,
            width=6,
            alpha=0.8,
            label=city
        )

    ax.set_title(title)
    ax.set_xlabel("Decade")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis="y")
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("Cumulative flood impact by decade: Lagos vs Durban",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "flood_impact_by_decade.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#STEP 4e- Flood summary table-Statistics

summary_rows = []

for city in ["Lagos","Durban"]:

    city_data = emdat[
        emdat["City"] == city
    ]

    row = {

        "City": city,

        "Total flood events":
            len(city_data),

        "First event":
            city_data["Year"].min(),

        "Last event":
            city_data["Year"].max(),

        "Total deaths":
            int(city_data["Deaths"].sum()),

        "Average deaths per event":
            round(city_data["Deaths"].mean(),2),

        "Worst event deaths":
            int(city_data["Deaths"].max())
    }

    if "Total_affected" in city_data.columns:

        row["People affected (millions)"] = round(
            city_data["Total_affected"].sum() / 1e6,
            2
        )

    if "Damage_million_usd" in city_data.columns:

        row["Total damage (Million USD)"] = round(
            city_data["Damage_million_usd"].sum(),
            2
        )

        row["Average damage/event"] = round(
            city_data["Damage_million_usd"].mean(),
            2
        )

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

print("\n=== FLOOD SUMMARY ===")
print(summary_df)

#Save to Excel
summary_df.to_excel(
    os.path.join(
        OUTPUT_DIR,
        "flood_summary_statistics.xlsx"
    ),
    index=False
)

print("\nSaved: flood_summary_statistics.xlsx")

In [ ]:
#STEP 4f- Mann-Kendall test on flood frequency 
#(same approach as Step 2e but applied to event counts)
import pymannkendall as mk

print("\n=== FLOOD FREQUENCY TREND TEST ===\n")

for city in ["Lagos","Durban"]:

    d = freq[
        freq["City"] == city
    ].sort_values("Year")

    result = mk.original_test(
        d["Flood_events"].values
    )

    print(city)
    print("Trend:", result.trend)
    print("p-value:", round(result.p,4))
    print("Tau:", round(result.Tau,3))
    print("Sen slope:", round(result.slope,4))
    print()

In [ ]:
emdat.to_csv("outputs/emdat_floods_clean.csv", index=False)

In [ ]:
#####Summary
##Main Outcomes
#Summarised historical flood events.
#Quantified changes in flood impacts.
#Evaluated economic losses through time.
#Prepared response variables for regression modelling.

##Limitations
#EM-DAT records include only events meeting the database inclusion criteria and may not capture all local flood events.

##Next Notebook
#->**05_regression_modelling.ipynb**
#The following notebook investigates statistical relationships between climate, population growth and flood impacts using ordinary least squares regression.